# *Nonlinear Arterial Hemodynamics*
## Chapter 2 companion — The Mechanical Limits of Wall Shear Stress

This notebook is the computational companion to Chapter 2. It reproduces the chapter's exact mechanical counterexample and uses VascuQuest to deepen the central question:

**What information is lost when a wall traction or a scalar wall-based projection is used as a surrogate for the neighboring fluid volume?**

The book nomenclature governs the notebook. Database-native names are confined to the ingestion/mapping layer.

The notebook admits the same mechanics as Chapter 2: Navier–Stokes mechanics, stress projection, velocity–vorticity interaction, the Gromeka–Lamb identity, and an endothelial-scale control volume. Constitutive anisotropy, geometry-specific spectral dynamics, wall compliance, and the Chapter 3 harmonic Womersley transfer solution are deliberately withheld.

**Execution:** a clean Google Colab runtime should reproduce all results with **Run all** and no manual parameter choices.

### Chapter question

Wall shear stress is exact where it is defined:

$$
\boldsymbol\tau_w
=
(\mathbf I-\mathbf n\mathbf n)\boldsymbol\sigma\mathbf n.
$$

The limitation appears only when this boundary quantity is asked to stand in for the full neighboring state.

The notebook therefore follows the chapter's own sequence:

1. reproduce the boundary-traction definition;
2. construct two distinct interior velocity fields with identical wall gradient and WSS;
3. compare the resulting volumetric vorticity and Lamb-vector fields;
4. integrate the Lamb-vector force-density contribution over an endothelial-scale near-wall pillbox;
5. reconstruct before multiplying when harmonic signals are involved;
6. use VascuQuest to show, descriptively, that a single scalar reference-state wall projection does not determine pulsatile waveform content.

The VascuQuest analysis is illustrative of information loss. It is not experimental validation of the chapter's mechanics.

### VascuQuest representation

VascuQuest supplies validated PWDB virtual-population waveforms and subject metadata.

For the Chapter 2 population analysis, the notebook uses:

- age;
- aortic-root flow-velocity waveform;
- aortic-root luminal-area waveform.

The source fields are combined internally to recover the book quantity

$$
Q(t)=U(t)A(t),
$$

where the source flow-velocity and area names remain confined to the mapping code.

A time-mean equivalent circular radius is obtained from the source luminal area,

$$
R=\sqrt{\frac{\langle A\rangle_t}{\pi}}.
$$

To create a **reference-state scalar projection only**, the steady Poiseuille relation from Chapter 1 is applied to the cycle-mean flow:

$$
\tau_{w,\mathrm{P}}
=
-\frac{4\mu\,\langle Q\rangle_t}{\pi R^3}.
$$

The subscript ``P'' in the code and prose denotes **Poiseuille-equivalent**. This is not claimed to be the actual pulsatile WSS of PWDB. The full harmonic Womersley relation required for that calculation belongs to Chapter 3.

The population exercise asks only whether nearly equal values of this scalar reference projection uniquely determine the underlying pulsatile waveform. They do not need to, and the analytical counterexample already establishes the more general many-to-one result.

Age is retained in the reproducibility data but is not promoted to a principal comparison here because age stratification does not sharpen the Chapter 2 mechanical argument.

In [ ]:
# Configuration and reproducibility constants
from pathlib import Path
import sys, json, subprocess, zipfile

ROOT = Path("/content/nonlinear_arterial_hemodynamics_ch02")
FIG_DIR = ROOT / "figures"
DATA_DIR = ROOT / "data"
META_DIR = ROOT / "metadata"
for d in (ROOT, FIG_DIR, DATA_DIR, META_DIR):
    d.mkdir(parents=True, exist_ok=True)

VQ_REPOSITORY = "https://github.com/KNOWDYN/VascuQuest.git"
VQ_GIT_REF = "8307147d72e7a6f3ea3135895bd6f52927c67439"
PWDB_RECORD_ID = "3275625"
PWDB_DOI = "10.5281/zenodo.3275625"

rho = 1060.0       # kg m^-3
mu = 3.5e-3        # Pa s
nu = mu / rho      # m^2 s^-1

THETA_EXAMPLE = 0.6
SITE = "AorticRoot"

print("Working directory:", ROOT)
print(f"nu = {nu:.6e} m^2/s")

In [ ]:
# Install the pinned VascuQuest revision.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    f"git+{VQ_REPOSITORY}@{VQ_GIT_REF}"
])

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import vascuquest as vq

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("matplotlib:", matplotlib.__version__)
print("VascuQuest:", getattr(vq, "__version__", "version field not exposed"))

In [ ]:
# Acquire only the PWDB artifacts required for this chapter.
ARTIFACTS = ["model_configurations", "common_site_waveforms_csv"]
verification = {}

for artifact in ARTIFACTS:
    subprocess.run(
        ["vascuquest", "dataset", "acquire",
         "--artifact", artifact, "--yes", "--format", "json"],
        check=True, text=True, capture_output=True,
    )
    verified = subprocess.run(
        ["vascuquest", "dataset", "verify",
         "--artifact", artifact, "--format", "json"],
        check=True, text=True, capture_output=True,
    )
    verification[artifact] = json.loads(verified.stdout)

status = subprocess.run(
    ["vascuquest", "dataset", "status", "--format", "json"],
    check=True, text=True, capture_output=True,
)
dataset_status = json.loads(status.stdout)
SOURCE_DIR = Path(dataset_status["managed_paths"]["source"])

(META_DIR / "artifact_verification.json").write_text(
    json.dumps(verification, indent=2), encoding="utf-8"
)
(META_DIR / "dataset_status.json").write_text(
    json.dumps(dataset_status, indent=2), encoding="utf-8"
)

print("Verified PWDB source:", SOURCE_DIR)

In [ ]:
# Open the verified dataset and retain subject metadata.
session = vq.open_dataset(source=SOURCE_DIR, offline=True)
assert session.identity.record_id == PWDB_RECORD_ID

age_result = session.get("age")
subject_ids = np.asarray(age_result.coordinates[0].values, dtype=str)
ages = np.asarray(age_result.values, dtype=float)

subject_meta = pd.DataFrame({
    "subject_id": subject_ids,
    "age_years": ages,
})
subject_meta["subject_number"] = subject_meta["subject_id"].astype(int)
subject_meta = subject_meta.sort_values("subject_number").reset_index(drop=True)

print("PWDB subjects:", len(subject_meta))
print("Source age strata:", sorted(subject_meta["age_years"].dropna().unique().tolist()))

In [ ]:
# Shared B&W plotting system and bulk waveform reader.
WAVE_ZIP = SOURCE_DIR / "PWs_csv.zip"

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["DejaVu Serif"],
    "mathtext.fontset": "stix",
    "font.size": 9.0,
    "axes.labelsize": 9.0,
    "axes.titlesize": 9.5,
    "xtick.labelsize": 8.0,
    "ytick.labelsize": 8.0,
    "legend.fontsize": 7.8,
    "axes.linewidth": 0.75,
    "lines.linewidth": 1.2,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})
BLACK, DARK, MID, LIGHT = "0.0", "0.28", "0.52", "0.74"

def clean_axes(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(False)

def save_figure(fig, stem):
    pdf = FIG_DIR / f"{stem}.pdf"
    png = FIG_DIR / f"{stem}.png"
    fig.savefig(pdf, bbox_inches="tight", pad_inches=0.03)
    fig.savefig(png, dpi=600, bbox_inches="tight", pad_inches=0.03)
    return pdf, png

def _wave_member_name(site_id, source_signal):
    basename = f"PWs_{site_id}_{source_signal}.csv"
    with zipfile.ZipFile(WAVE_ZIP, "r") as zf:
        matches = [name for name in zf.namelist() if Path(name).name == basename]
    if len(matches) != 1:
        raise RuntimeError(f"Expected one {basename!r}; found {len(matches)}")
    return matches[0]

def load_waveform_matrix(site_id, source_signal):
    # Native source signal names are isolated here.
    member = _wave_member_name(site_id, source_signal)
    with zipfile.ZipFile(WAVE_ZIP, "r") as zf:
        with zf.open(member, "r") as raw:
            frame = pd.read_csv(raw, low_memory=False)
    ids = np.asarray([str(int(x)) for x in frame.iloc[:, 0].to_numpy()], dtype=str)
    values = frame.iloc[:, 1:].to_numpy(dtype=float)
    return ids, values

print("Shared helpers ready.")

# Boundary traction and volumetric state

For an incompressible Newtonian fluid,

$$
\boldsymbol\sigma=-p\mathbf I+2\mu\mathbf D,
\qquad
\mathbf D
=
\frac12
\left(
\nabla\mathbf u+\nabla\mathbf u^{\mathsf T}
\right).
$$

The Cauchy traction is

$$
\mathbf t(\mathbf n)=\boldsymbol\sigma\mathbf n,
$$

and the tangential wall traction is

$$
\boldsymbol\tau_w
=
(\mathbf I-\mathbf n\mathbf n)
\boldsymbol\sigma\mathbf n.
$$

In the straight-tube state,

$$
\boldsymbol\tau_w
=
\mu
\left.
\frac{\partial u_z}{\partial r}
\right|_R
\mathbf e_z.
$$

Appendix C supplies the continuum-mechanical traction hierarchy; Appendix D supplies the Newtonian constitutive relation. The notebook begins from those definitions rather than from a biological WSS metric.

# A constructive counterexample

Let

$$
x=\frac{r}{R},
$$

and define

$$
u_A(r)=U(1-x^2),
$$

$$
u_B(r)=
U\left[
1-x^2+\vartheta(1-x^2)^2
\right].
$$

Both satisfy no slip and regularity at the centreline. At the wall,

$$
\left.
\frac{du_A}{dr}
\right|_R
=
\left.
\frac{du_B}{dr}
\right|_R
=
-\frac{2U}{R},
$$

so

$$
\tau_{w,A}=\tau_{w,B}
=
-\frac{2\mu U}{R}.
$$

But

$$
Q_A=\frac{\pi R^2U}{2},
$$

whereas

$$
Q_B
=
\pi R^2U
\left(
\frac12+\frac{\vartheta}{3}
\right).
$$

The mapping from interior field to WSS is therefore many-to-one even in this elementary setting.

In [ ]:
# Reproduce the Chapter 2 counterexample and extend it to the interior shear field.
# Cylindrical derivatives follow Appendix B.
x = np.linspace(0.0, 1.0, 600)
theta = THETA_EXAMPLE

uA_over_U = 1.0 - x**2
uB_over_U = 1.0 - x**2 + theta * (1.0 - x**2)**2

# R/U times du/dr: dimensionless radial gradients.
gradA = -2.0 * x
gradB = -2.0 * x - 4.0 * theta * x * (1.0 - x**2)

fig, axes = plt.subplots(1, 2, figsize=(7.0, 3.0))

axes[0].plot(x, uA_over_U, color=BLACK, linestyle="-", label=r"$u_A/U$")
axes[0].plot(x, uB_over_U, color=DARK, linestyle="--", label=r"$u_B/U$")
axes[0].set_xlabel(r"Normalized radius, $x=r/R$")
axes[0].set_ylabel("Normalized axial velocity")
axes[0].legend(frameon=False)
clean_axes(axes[0])

axes[1].plot(x, gradA, color=BLACK, linestyle="-",
             label=r"$(R/U)\,du_A/dr$")
axes[1].plot(x, gradB, color=DARK, linestyle="--",
             label=r"$(R/U)\,du_B/dr$")
axes[1].axvline(1.0, color=LIGHT, linewidth=0.8)
axes[1].set_xlabel(r"Normalized radius, $x=r/R$")
axes[1].set_ylabel("Normalized radial gradient")
axes[1].legend(frameon=False)
clean_axes(axes[1])

fig.tight_layout(w_pad=1.3)
save_figure(fig, "ch02_same_wss_different_fields")
plt.show()

The right panel exposes the exact source of the non-uniqueness: the two radial-gradient fields differ in the interior but meet at the same wall value.

The WSS equality is therefore exact, while the interior difference is not a perturbation or numerical artifact.

# Returning to the nonlinear momentum equation

The incompressible Navier–Stokes equation is

$$
\frac{\partial\mathbf u}{\partial t}
+
(\mathbf u\cdot\nabla)\mathbf u
=
-\frac{1}{\rho}\nabla p
+
\nu\nabla^2\mathbf u.
$$

Using the Gromeka–Lamb identity,

$$
(\mathbf u\cdot\nabla)\mathbf u
=
\nabla\left(\frac{|\mathbf u|^2}{2}\right)
-
\mathbf u\times\boldsymbol\omega,
$$

with

$$
\boldsymbol\omega=\nabla\times\mathbf u,
$$

define

$$
\boldsymbol\ell
=
\mathbf u\times\boldsymbol\omega.
$$

Appendix B supplies the cylindrical curl and convective-acceleration identities. The field $\rho\boldsymbol\ell$ has units of force per unit volume.

In [ ]:
# For fully developed axial flow, omega = -(du_z/dr) e_theta and
# ell = u_z (du_z/dr) e_r, exactly as derived in Chapter 2.
#
# Normalize ell_r by U^2/R so that the comparison is independent of the
# arbitrary dimensional scale chosen for the constructive counterexample.
ellA_scaled = uA_over_U * gradA
ellB_scaled = uB_over_U * gradB

# The kinetic-energy gradient has the same radial value in this kinematic limit.
kegradA_scaled = ellA_scaled.copy()
kegradB_scaled = ellB_scaled.copy()

fig, axes = plt.subplots(1, 2, figsize=(7.0, 3.0))

axes[0].plot(x, ellA_scaled, color=BLACK, linestyle="-",
             label=r"profile $u_A$")
axes[0].plot(x, ellB_scaled, color=DARK, linestyle="--",
             label=r"profile $u_B$")
axes[0].set_xlabel(r"Normalized radius, $x=r/R$")
axes[0].set_ylabel(r"$(R/U^2)\,\ell_r$")
axes[0].legend(frameon=False)
clean_axes(axes[0])

axes[1].plot(x, kegradA_scaled - ellA_scaled, color=BLACK, linestyle="-",
             label=r"profile $u_A$")
axes[1].plot(x, kegradB_scaled - ellB_scaled, color=DARK, linestyle="--",
             label=r"profile $u_B$")
axes[1].set_xlabel(r"Normalized radius, $x=r/R$")
axes[1].set_ylabel("Normalized convective acceleration")
axes[1].set_ylim(-0.03, 0.03)
axes[1].legend(frameon=False)
clean_axes(axes[1])

fig.tight_layout(w_pad=1.3)
save_figure(fig, "ch02_lamb_fields_and_cancellation")
plt.show()

# The classical Womersley limit exposes the distinction

For

$$
\mathbf u=u_z(r,t)\mathbf e_z,
$$

the vorticity and Lamb vector are

$$
\boldsymbol\omega
=
-\frac{\partial u_z}{\partial r}\mathbf e_\theta,
$$

$$
\boldsymbol\ell
=
u_z\frac{\partial u_z}{\partial r}\mathbf e_r.
$$

At the same time,

$$
\nabla\left(\frac{|\mathbf u|^2}{2}\right)
=
u_z\frac{\partial u_z}{\partial r}\mathbf e_r.
$$

Hence

$$
\nabla\left(\frac{|\mathbf u|^2}{2}\right)
-
\boldsymbol\ell
=
\mathbf0.
$$

The notebook calculation above makes the key point visible: a finite Lamb-vector field can exist while the complete convective acceleration remains zero because the kinetic-energy gradient cancels it exactly.

This is a statement about the exact identity, not evidence of a new transverse flow mechanism.

# Information retained by wall projection

The counterexample can be extended from one value of $\vartheta$ to a complete one-parameter family.

The wall gradient and WSS remain unchanged for every $\vartheta$, while the flow rate changes according to

$$
\frac{Q_B}{Q_A}
=
1+\frac{2\vartheta}{3}.
$$

The same family also changes the magnitude and spatial distribution of the velocity–vorticity product. A single wall derivative therefore leaves a continuum of admissible interior states.

In [ ]:
# Family-level information loss: WSS remains fixed while other quantities vary.
theta_values = np.linspace(-0.45, 1.5, 250)
Q_ratio = 1.0 + (2.0 / 3.0) * theta_values

# Compute a cross-sectional magnitude measure of the Lamb-vector field.
# The factor x is the cylindrical area measure; constants 2*pi*R^2 cancel
# in the ratio. This is a dimensionless diagnostic, not a new book symbol.
x_int = np.linspace(0.0, 1.0, 2000)

def lamb_magnitude_integral(theta_value):
    u = 1.0 - x_int**2 + theta_value * (1.0 - x_int**2)**2
    grad = -2.0 * x_int - 4.0 * theta_value * x_int * (1.0 - x_int**2)
    return np.trapz(np.abs(u * grad) * x_int, x_int)

baseline = lamb_magnitude_integral(0.0)
lamb_ratio = np.array([
    lamb_magnitude_integral(th) / baseline
    for th in theta_values
])

fig, axes = plt.subplots(1, 2, figsize=(7.0, 3.0))

axes[0].plot(theta_values, Q_ratio, color=BLACK)
axes[0].axhline(1.0, color=LIGHT, linestyle=":", linewidth=0.9)
axes[0].set_xlabel(r"Profile parameter, $\vartheta$")
axes[0].set_ylabel(r"$Q_B/Q_A$")
clean_axes(axes[0])

axes[1].plot(theta_values, lamb_ratio, color=BLACK)
axes[1].axhline(1.0, color=LIGHT, linestyle=":", linewidth=0.9)
axes[1].set_xlabel(r"Profile parameter, $\vartheta$")
axes[1].set_ylabel("Relative cross-sectional\nLamb-vector magnitude integral")
clean_axes(axes[1])

fig.tight_layout(w_pad=1.4)
save_figure(fig, "ch02_many_to_one_family")
plt.show()

The unchanged WSS is not plotted because it is identically the same for the entire family. The two panels instead show quantities that remain free to vary after that wall information has been fixed.

This is the computational form of the chapter statement that wall projection is lossy while the wall traction itself remains mechanically exact.

# Integral momentum and the status of a force descriptor

For a fixed control volume $V$ with boundary $S$,

$$
\frac{d}{dt}
\int_V\rho\mathbf u\,dV
+
\int_S
\rho\mathbf u(\mathbf u\cdot\mathbf n)\,dA
=
\int_S
\boldsymbol\sigma\mathbf n\,dA
+
\int_V
\rho\mathbf b\,dV.
$$

The volume integral

$$
\mathbf F_{\ell,V}(t)
=
\int_V
\rho\boldsymbol\ell(\mathbf x,t)\,dV
$$

is therefore a well-defined integrated descriptor with units of force, but it is not automatically the net hydrodynamic force on a solid boundary.

Appendix C provides the control-volume momentum balance that fixes this distinction.

# The endothelial-scale pillbox

Let

$$
\delta_{\mathrm{EC}}
=
\frac{V_{\mathrm{EC}}}{A_{\mathrm{EC}}}.
$$

For a locally cylindrical wall, the near-wall descriptor is

$$
F_{r,\mathrm{EC}}^{\mathrm{mag}}(t)
=
A_{\mathrm{EC}}
\int_{R-\delta_{\mathrm{EC}}}^{R}
\left|
\rho\ell_r(r,t)
\right|\,dr.
$$

The book deliberately separates this magnitude-accumulation descriptor from a signed net force.

No numerical endothelial-cell dimensions are invented here. Instead, the computation uses the dimensionless geometric ratio $\delta_{\mathrm{EC}}/R$ so the mechanical dependence can be examined without adding biological constants not specified by the chapter.

In [ ]:
# Near-wall pillbox sensitivity for profiles that all share the same WSS.
delta_ratio = np.linspace(0.002, 0.20, 180)
theta_set = [0.0, 0.3, 0.6, 1.0]
styles = [
    (BLACK, "-", "profile A"),
    (DARK, "--", r"$\vartheta=0.3$"),
    (MID, "-.", r"$\vartheta=0.6$"),
    (LIGHT, ":", r"$\vartheta=1.0$"),
]

def near_wall_lamb_integral(theta_value, d_over_R):
    # Normalize by rho*A_EC*U^2. The remaining integral is dimensionless.
    r0 = max(0.0, 1.0 - d_over_R)
    xx = np.linspace(r0, 1.0, 800)
    u = 1.0 - xx**2 + theta_value * (1.0 - xx**2)**2
    grad = -2.0 * xx - 4.0 * theta_value * xx * (1.0 - xx**2)
    return np.trapz(np.abs(u * grad), xx)

fig, ax = plt.subplots(figsize=(5.6, 3.3))

for th, (gray, ls, label) in zip(theta_set, styles):
    values = np.array([
        near_wall_lamb_integral(th, d)
        for d in delta_ratio
    ])
    ax.plot(delta_ratio, values, color=gray, linestyle=ls, label=label)

ax.set_xlabel(r"Normalized pillbox thickness, $\delta_{\mathrm{EC}}/R$")
ax.set_ylabel(r"$F_{r,\mathrm{EC}}^{\mathrm{mag}}/(\rho A_{\mathrm{EC}}U^2)$")
ax.legend(frameon=False)
clean_axes(ax)
fig.tight_layout()

save_figure(fig, "ch02_endothelial_pillbox")
plt.show()

All curves in the pillbox plot correspond to profiles with the **same wall gradient and the same Newtonian WSS**. Their near-wall integrated Lamb-vector magnitude nevertheless differs.

This does not imply that the pillbox descriptor replaces WSS. It demonstrates that the two operations answer different mechanical questions:

- WSS evaluates tangential traction at the boundary;
- the pillbox descriptor integrates a selected volumetric inertial contribution through a finite near-wall fluid region.

# Why harmonic solutions must be reconstructed before multiplication

For a multiharmonic velocity,

$$
\mathbf u(\mathbf x,t)
=
\Re\left\{
\sum_{m=1}^{M}
\widehat{\mathbf u}_m(\mathbf x)
e^{im\Omega t}
\right\},
$$

and

$$
\boldsymbol\omega(\mathbf x,t)
=
\Re\left\{
\sum_{n=1}^{M}
\widehat{\boldsymbol\omega}_n(\mathbf x)
e^{in\Omega t}
\right\},
$$

the nonlinear field

$$
\boldsymbol\ell(t)
=
\mathbf u(t)\times\boldsymbol\omega(t)
$$

contains cross-harmonic products.

Therefore the physical time signals must be reconstructed before multiplication, or the equivalent frequency-domain convolution must be evaluated. Multiplying only matching complex amplitudes omits sum- and difference-frequency interactions.

The following deterministic demonstration uses a two-harmonic scalar analogue of the same bilinear structure. It does not introduce the Chapter 3 Womersley transfer function.

In [ ]:
# Minimal reconstruction-before-multiplication demonstration.
phase = np.linspace(0.0, 2.0*np.pi, 1200, endpoint=False)

# Two independently linear harmonic signals.
u_signal = np.cos(phase) + 0.45*np.cos(2.0*phase - 0.4)
omega_signal = 0.8*np.cos(phase + 0.7) + 0.30*np.cos(2.0*phase + 0.2)

# Bilinear observable after reconstructing the real signals.
product_signal = u_signal * omega_signal
coeff = np.fft.rfft(product_signal) / len(product_signal)
m = np.arange(len(coeff))
amp = 2.0 * np.abs(coeff)
amp[0] = np.abs(coeff[0])

fig, ax = plt.subplots(figsize=(5.6, 3.2))
ax.stem(m[:6], amp[:6], linefmt="k-", markerfmt="ko", basefmt=" ")
ax.set_xlabel(r"Harmonic number, $m$")
ax.set_ylabel("Product-spectrum amplitude")
ax.set_xticks(m[:6])
clean_axes(ax)
fig.tight_layout()

save_figure(fig, "ch02_cross_harmonic_products")
plt.show()

The product contains a mean contribution and additional sum- and difference-frequency content even though the two input fields were built from only the first two harmonics.

This is the computational reason that Chapter 2 requires reconstruction before multiplication when evaluating $\mathbf u\times\boldsymbol\omega$ from linear harmonic solutions.

# VascuQuest exploration: scalar projection versus pulsatile state

The analytical counterexample already proves the many-to-one character of wall projection.

VascuQuest is now used for a narrower empirical question:

> Within one arterial site, can two virtual subjects have nearly the same **Poiseuille-equivalent cycle-mean wall-shear projection** while retaining materially different pulsatile $Q(t)$ content?

The reference projection is

$$
\tau_{w,\mathrm{P}}
=
-\frac{4\mu\,\langle Q\rangle_t}{\pi R^3}.
$$

Again, this is not claimed to be actual pulsatile WSS. Its purpose is to create one scalar boundary-style reference quantity from the Chapter 1 Poiseuille relation and test how much waveform information that scalar retains.

In [ ]:
# Build the aortic-root population projection directly from the verified PWDB archive.
ids_u, U_matrix = load_waveform_matrix(SITE, "U")
ids_a, A_matrix = load_waveform_matrix(SITE, "A")
assert np.array_equal(ids_u, ids_a)

rows = []
waveforms = {}

for sid, U_row, A_row in zip(ids_u, U_matrix, A_matrix):
    valid = np.isfinite(U_row) & np.isfinite(A_row)
    if valid.sum() < 16:
        continue

    U_values = U_row[valid]
    A_values = A_row[valid]
    Q_values = U_values * A_values

    mean_area = float(np.mean(A_values))
    R_value = float(np.sqrt(mean_area / np.pi))
    mean_Q = float(np.mean(Q_values))

    # Poiseuille-equivalent cycle-mean wall shear: reference projection only.
    tau_w_P = -4.0 * mu * mean_Q / (np.pi * R_value**3)

    # Quantify how much periodic content remains outside the mean.
    coeff = np.fft.rfft(Q_values) / len(Q_values)
    total_energy = float(np.sum(np.abs(coeff)**2))
    nonmean_energy = float(np.sum(np.abs(coeff[1:])**2))
    nonmean_fraction = nonmean_energy / total_energy if total_energy > 0 else np.nan

    age_match = subject_meta.loc[subject_meta["subject_id"] == sid, "age_years"]
    age_years = float(age_match.iloc[0]) if len(age_match) else np.nan

    rows.append({
        "subject_id": sid,
        "age_years": age_years,
        "R_m": R_value,
        "mean_Q_m3_s": mean_Q,
        "tau_w_P_Pa": tau_w_P,
        "nonmean_harmonic_energy_fraction": nonmean_fraction,
    })
    waveforms[sid] = Q_values

projection_df = pd.DataFrame(rows).dropna().reset_index(drop=True)
projection_df.to_csv(DATA_DIR / "ch02_scalar_projection_population.csv", index=False)

# Deterministic pair search:
# nearly equal |tau_w,P| (within 0.5%) and maximal difference in waveform harmonic content.
work = projection_df.copy()
work["abs_tau"] = work["tau_w_P_Pa"].abs()
work = work.sort_values(["abs_tau", "subject_id"]).reset_index(drop=True)

best = None
for i in range(len(work)):
    tau_i = work.loc[i, "abs_tau"]
    if tau_i <= 0:
        continue
    for j in range(i + 1, min(i + 80, len(work))):
        tau_j = work.loc[j, "abs_tau"]
        rel_diff = abs(tau_j - tau_i) / max(tau_i, tau_j)
        if rel_diff > 0.005:
            if tau_j > tau_i:
                break
            continue
        contrast = abs(
            work.loc[j, "nonmean_harmonic_energy_fraction"]
            - work.loc[i, "nonmean_harmonic_energy_fraction"]
        )
        candidate = (contrast, -rel_diff, i, j)
        if best is None or candidate > best:
            best = candidate

if best is None:
    raise RuntimeError("No deterministic near-equal projection pair found.")

_, _, i_best, j_best = best
pair = work.loc[[i_best, j_best]].copy().reset_index(drop=True)
pair.to_csv(DATA_DIR / "ch02_selected_projection_pair.csv", index=False)

display(pair)

In [ ]:
# Plot the population compression map and the selected nearly-equal pair.
sid1, sid2 = pair.loc[0, "subject_id"], pair.loc[1, "subject_id"]

fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.15))

axes[0].scatter(
    projection_df["tau_w_P_Pa"].abs(),
    projection_df["nonmean_harmonic_energy_fraction"],
    s=9, facecolors="none", edgecolors=MID, linewidths=0.55,
)
axes[0].scatter(
    pair["tau_w_P_Pa"].abs(),
    pair["nonmean_harmonic_energy_fraction"],
    s=34, facecolors="white", edgecolors=BLACK, linewidths=1.0,
)
axes[0].set_xlabel(r"$|\tau_{w,\mathrm{P}}|$ (Pa)")
axes[0].set_ylabel("Non-mean harmonic energy fraction")
clean_axes(axes[0])

for sid, gray, ls, label in [
    (sid1, BLACK, "-", "selected subject 1"),
    (sid2, DARK, "--", "selected subject 2"),
]:
    q = np.asarray(waveforms[sid], dtype=float)
    phase_q = np.arange(len(q), dtype=float) / len(q)
    scale = np.max(np.abs(q))
    axes[1].plot(
        phase_q, q / scale,
        color=gray, linestyle=ls, label=label
    )

axes[1].set_xlabel(r"Normalized time, $t/T$")
axes[1].set_ylabel(r"$Q(t)/\max_t|Q(t)|$")
axes[1].legend(frameon=False)
clean_axes(axes[1])

fig.tight_layout(w_pad=1.3)
save_figure(fig, "ch02_vascuquest_projection_information_loss")
plt.show()

The left panel is a **VascuQuest population observation**. The scalar reference projection occupies one axis, while the remaining pulsatile harmonic content occupies another. A single scalar does not determine that temporal structure.

The highlighted pair is selected deterministically from the population by requiring the Poiseuille-equivalent wall-shear magnitudes to agree within 0.5% and then maximizing the difference in non-mean harmonic content among such pairs. The right panel shows their normalized $Q(t)$ waveforms.

The conclusion is deliberately limited:

- this figure does **not** show that the two subjects have identical actual pulsatile WSS;
- it does show that an equal scalar Poiseuille-equivalent wall projection does not uniquely specify the pulsatile state from which it was constructed;
- the chapter's stronger mechanical non-uniqueness is already established exactly by the analytical profiles $u_A$ and $u_B$.

# Mechanical questions resolved at the wall and in the volume

The notebook separates the mechanical questions in the same way as the chapter.

| Quantity or operation | Mechanical information retained |
|---|---|
| $\boldsymbol\tau_w$ | tangential traction at the wall |
| $u_z(r,t)$ | interior velocity field |
| $\boldsymbol\omega$ | interior rotational structure |
| $\boldsymbol\ell=\mathbf u\times\boldsymbol\omega$ | local velocity–vorticity coupling |
| $\rho\boldsymbol\ell$ | corresponding force-density contribution |
| $\int_V\rho\boldsymbol\ell\,dV$ | volume-integrated descriptor |
| $F_{r,\mathrm{EC}}^{\mathrm{mag}}$ | accumulated near-wall radial Lamb-vector magnitude over the endothelial-scale pillbox |
| scalar or time-averaged projection | reduced information; phase and harmonic content are not recoverable unless retained separately |

WSS remains exact for wall loading under the specified constitutive law. The volumetric quantities answer different questions and are not recoverable uniquely from that boundary projection alone.

# What the reader should learn

1. **WSS is not mechanically deficient at the wall.** It is the correct tangential traction there.

2. **The limitation is representational.** Distinct interior fields can have the same wall gradient and therefore the same Newtonian WSS.

3. **The information loss is visible in exact equations.** The family parameter $\vartheta$ can change $Q$, vorticity, and the Lamb-vector field while leaving WSS unchanged.

4. **A finite Lamb-vector field does not by itself imply nonzero convective acceleration.** In the fully developed classical limit, the kinetic-energy gradient cancels it exactly.

5. **The endothelial-scale pillbox is a volume integration operation, not a redefinition of wall force.** It isolates a selected near-wall inertial contribution while the complete momentum balance retains authority for net force.

6. **Quadratic observables must be formed after real-field reconstruction or by an equivalent convolution.** Linear harmonic solutions do not imply harmonic-by-harmonic multiplication of nonlinear observables.

7. **VascuQuest reinforces the information-compression point without being used as proof of the mechanics.** Nearly equal scalar reference projections can coexist with different pulsatile waveform content.

# Chapter-enrichment candidates

The notebook produces five principal figures.

**Candidate 1 — same WSS, different velocity and radial-gradient fields.**  
This reproduces and slightly deepens the chapter's existing constructive counterexample. Because Chapter 2 already contains a same-WSS figure, replacement rather than addition should be considered if the notebook rendering is materially clearer.

**Candidate 2 — Lamb-vector fields and exact Gromeka–Lamb cancellation.**  
Strong candidate if the chapter needs a direct visual distinction between a finite $\boldsymbol\ell$ field and zero net convective acceleration in the fully developed limit.

**Candidate 3 — one-parameter many-to-one family.**  
Potentially strong because it shows continuously, rather than through one selected $\vartheta$, how WSS can remain fixed while flow and volumetric information vary.

**Candidate 4 — endothelial pillbox sensitivity.**  
Strong candidate if the pillbox construction benefits from a quantitative visual showing that equal WSS does not imply equal near-wall integrated Lamb-vector magnitude.

**Candidate 5 — VascuQuest scalar-projection information loss.**  
Primarily a notebook figure. It adds population context, but the analytical counterexample is mechanically stronger and should remain primary in the book.

The cross-harmonic demonstration is principally pedagogical notebook material unless Chapter 2 requires a compact visual for the reconstruction-before-multiplication rule.

In [ ]:
# Reproducibility record.
manifest = {
    "book": "Nonlinear Arterial Hemodynamics",
    "chapter": 2,
    "chapter_title": "The Mechanical Limits of Wall Shear Stress",
    "vascuquest_git_ref": VQ_GIT_REF,
    "pwdb_record_id": PWDB_RECORD_ID,
    "pwdb_doi": PWDB_DOI,
    "rho_kg_m3": rho,
    "mu_Pa_s": mu,
    "nu_m2_s": nu,
    "constructive_counterexample_theta": THETA_EXAMPLE,
    "vascuquest_site": SITE,
    "population_subject_count": int(len(projection_df)),
    "selected_pair_subject_ids": pair["subject_id"].astype(str).tolist(),
    "selected_pair_projection_relative_difference": float(
        abs(pair.loc[0, "tau_w_P_Pa"] - pair.loc[1, "tau_w_P_Pa"])
        / max(abs(pair.loc[0, "tau_w_P_Pa"]), abs(pair.loc[1, "tau_w_P_Pa"]))
    ),
    "python": sys.version.split()[0],
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "matplotlib": matplotlib.__version__,
    "execution_status": "completed to this cell",
    "qualification": (
        "The Poiseuille-equivalent wall-shear quantity is a Chapter 1 reference-state "
        "projection applied to PWDB cycle-mean flow. It is not represented as actual "
        "pulsatile WSS or experimental validation."
    ),
}

(META_DIR / "reproducibility_manifest.json").write_text(
    json.dumps(manifest, indent=2), encoding="utf-8"
)

print(json.dumps(manifest, indent=2))
print("\nGenerated PDF figures:")
for path in sorted(FIG_DIR.glob("*.pdf")):
    print(" -", path.name)

# Reproducibility record

A successful **Run all** execution writes:

- B&W vector PDF figures and high-resolution PNG previews;
- population data used in the VascuQuest projection analysis;
- the deterministically selected near-equal scalar-projection pair;
- VascuQuest/PWDB verification metadata;
- the final reproducibility manifest.

The notebook contains no interactive branch, manual parameter choice, or hidden hand-picked subject.